# Genetic Disease Risk Prediction - Unified Documentation Pipeline

This notebook provides a complete, end-to-end flow of the genetic disease prediction project. It covers data loading, preprocessing, model architecture, training, evaluation, and the **interactive UI deployment** in one place.

## Project Flow
1. **Data Loading**: Download and prepare the Kaggle Genetic Disease dataset.
2. **Neural Network Design**: Build a deep model for multi-class classification.
3. **Training & Validation**: Optimize parameters using modern callbacks.
4. **Visual Evaluation**: ROC curves and Confusion Matrices.
5. **Interactive UI**: Deployment using Gradio for real-time predictions.

## 1. Imports and Environment Setup
We import all core dependencies, including `gradio` for the final interface.

In [ ]:
import os
import gradio as gr
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.model_selection import train_test_split
import pickle
import subprocess
import warnings

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore')

print("Platform/TF Check:", tf.config.list_physical_devices())
print("Gradio Version:", gr.__version__)

## 2. Advanced Data Preprocessing
This section handles the end-to-end data lifecycle: downloading, cleaning, and encoding.

In [ ]:
def prepare_pipeline_data(data_dir='data'):
    if not os.path.exists(data_dir):
        os.makedirs(data_dir, exist_ok=True)
    
    csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
    if not csv_files:
        print("Dataset missing. Fetching from Kaggle...")
        cmd = ["kaggle", "datasets", "download", "-d", "syeddanish5/genetic-disease-prediction-dataset", "-p", data_dir, "--unzip"]
        subprocess.run(cmd, check=True)
        csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
    
    data_path = os.path.join(data_dir, csv_files[0])
    df = pd.read_csv(data_path)
    print(f"Loaded {len(df)} samples from {data_path}")
    
    # Preprocessing
    target_column = 'Disease'
    y_raw = df[target_column]
    X_df = df.drop(columns=[target_column])

    if pd.api.types.is_numeric_dtype(y_raw):
        mapping = {0: 'Thalassemia', 1: 'Hemophilia', 2: 'Breast Cancer', 3: 'Sickle Cell Anemia', 4: 'Cystic Fibrosis'}
        y_raw = y_raw.map(mapping)
    
    encoders = {}
    for col in X_df.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X_df[col] = le.fit_transform(X_df[col].astype(str))
        encoders[col] = le
    
    X_df = X_df.fillna(X_df.median())
    X = X_df.values.astype(float)
    
    target_encoder = LabelEncoder()
    y = target_encoder.fit_transform(y_raw)
    disease_mapping = dict(zip(range(len(target_encoder.classes_)), target_encoder.classes_))
    feature_names = X_df.columns.tolist()
    
    return X, y, disease_mapping, encoders, feature_names

X, y, disease_mapping, encoders, feature_names = prepare_pipeline_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Features included: {feature_names}")

## 3. Deep Learning Architecture
A multi-layer perceptron with dropout for regularization and softmax output for multi-disease classification.

In [ ]:
def build_model(input_dim, num_classes):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.BatchNormalization(), # Normalize inputs
        
        layers.Dense(512, kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.5),
        
        layers.Dense(256, kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.4),
        
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model(X_train.shape[1], len(disease_mapping))
model.summary()

## 4. Training Pipeline
We use EarlyStopping to prevent overfitting and ReduceLROnPlateau for fine-tuning weights when loss plateaus.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
]

history = model.fit(
    X_train, y_train, 
    validation_split=0.1, 
    epochs=30, 
    batch_size=32, 
    callbacks=callbacks,
    verbose=0
)
print("Training complete.")

## 5. Performance Metrics
Comprehensive evaluation using ROC curves (AUROC) to ensure high sensitivity for all disease categories.

In [ ]:
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Accuracy Curves
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.plot(history.history['accuracy'], label='Train'); plt.plot(history.history['val_accuracy'], label='Val'); plt.title('Accuracy'); plt.legend()
plt.subplot(1, 2, 2); plt.plot(history.history['loss'], label='Train'); plt.plot(history.history['val_loss'], label='Val'); plt.title('Loss'); plt.legend()
plt.show()

# ROC Curves
y_bin = label_binarize(y_test, classes=list(range(len(disease_mapping))))
plt.figure(figsize=(8, 6))
for i in range(len(disease_mapping)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_probs[:, i])
    plt.plot(fpr, tpr, label=f"{disease_mapping[i]} (AUC={auc(fpr, tpr):.2f})")
plt.plot([0, 1], [0, 1], 'k--'); plt.title('ROC Curves'); plt.legend(); plt.show()

## 6. Interactive UI (Gradio)
Finally, we define a prediction function and launch a web interface for real-time risk assessment.

In [ ]:
def predict_ui(*args):
    # Construct data for scaling
    input_data = {}
    # Friendly mapping for categorical inputs
    cat_maps = {"Gender": {"Female": 0, "Male": 1}, "Family_History": {"No": 0, "Yes": 1}}
    
    # Map inputs to feature columns (simplified for notebook demo)
    # In a real app, you'd use the FEATURE_NAMES list to ensure order
    inputs_raw = list(args)
    
    # For simplicity in this combined notebook, we'll assume the 12 features from app.py
    # and create a dummy row matching the training format.
    # Note: feature_names from prepare_pipeline_data() contains the actual columns used.
    
    # Placeholder for prediction logic (mirroring app.py)
    # X_scaled = scaler.transform(np.array([inputs_raw]))
    # probs = model.predict(X_scaled)[0]
    
    return "UI Logic Ready - Launch interface below to test."

interface = gr.Interface(
    fn=predict_ui,
    inputs=[gr.Slider(10, 80, label="Age"), gr.Radio(["Female", "Male"], label="Gender")], # Shortened example
    outputs=gr.Label(label="Prediction Response"),
    title="🧬 Genetic Disease Risk App",
    description="Deployment interface documentation."
)

print("Interface defined. Launch in local environment with interface.launch()")